<a href="https://colab.research.google.com/github/hridhikjal-sb/pdg_chat_bot/blob/main/LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
!pip install -q --upgrade langchain langchain_google_genai langchain-core langchain_community docs2txt pypdf langchain_chroma sentence_transformers

In [41]:
from dotenv import load_dotenv
load_dotenv()  # defaults to loading from .env file

import os

In [42]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm=ChatGoogleGenerativeAI(model="gemini-1.5-flash")


!pip install langchain_communitym

In [43]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [44]:
#loading and split data

# Import necessary modules
import os
from langchain_community.document_loaders import PyPDFLoader,Docx2txtLoader # Loaders for reading PDF and Word files
from langchain_text_splitters import RecursiveCharacterTextSplitter# For splitting long texts into chunks
from typing import List
from langchain_core.documents import Document

# Initialize a text splitter that:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200,length_function=len)

# Define a function to load all .pdf and .docx files from a folder
def load_documents(folder_path : str)-> List[Document]:
  documents=[]
  for filename in os.listdir(folder_path):
    file_path=os.path.join(folder_path,filename)
    if filename.endswith('.pdf'):
      loader=PyPDFLoader(file_path)
    elif filename.endswith('.docx'):
      loader=Docx2txtLoader(file_path)
    else:
      print(f"unsupported file type : {filename}")
      continue
    documents.extend(loader.load())
  return documents
# Path to your documents folder (Google Drive in this case)
folder_path='/content/drive/MyDrive/project/LLM CHAT  FOR DOCS /docs'
documents=load_documents(folder_path)
print(f"loaded {len(documents)} from folder")

splits=text_splitter.split_documents(documents)
print(f"splitted documents into {len(splits)} chunks")


loaded 4 from folder
splitted documents into 4 chunks


In [45]:
#creating embedding out side langchain manually
from posixpath import split
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embedding=GoogleGenerativeAIEmbeddings(model="models/embedding-001")
document_embedding = embedding.embed_documents([split.page_content for split in splits])
print(f"created embedding for {len(document_embedding)} document chunks" )

created embedding for 4 document chunks


In [46]:
#Setting uo vector stor chromadb and embeding

from langchain_chroma import Chroma
# Set up the embedding model using Google's Generative AI embeddings
embedding_function = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
# Create and persist a Chroma vector store from the document chunks (splits)
collection_name ="my_collection"
vectorstore =Chroma.from_documents(
    collection_name =collection_name,
    documents=splits,
    embedding=embedding_function,
    persist_directory="./chroma_db"
)

print("vector store created and persisted to  '.chroma_db'" )

vector store created and persisted to  '.chroma_db'


In [47]:
query = "what is the resume summary?"

search_results =vectorstore.similarity_search(query,k=2)
print(f"\n top 2 most relevant search for the query :  '{query}' \n")
for i,result in enumerate(search_results,1):
  print(f"result {i} : \n")
  print(f"source: {result.metadata['source']}")
  print(f"content: {result.page_content}")
  print()


 top 2 most relevant search for the query :  'what is the resume summary?' 

result 1 : 

source: /content/drive/MyDrive/project/LLM CHAT  FOR DOCS /docs/HRIDHIKJAL SB.pdf
content: • B.Sc. in Mathematics – SN College, Kannur, Kerala (May 2020) 
• Higher Secondary (Science) – PRMHSS, Kannur, Kerala (May 2018) 
 
 
Certifications 
• Data Science – NACTET (Aug 2024) 
• Business Analysis & Process Management – Coursera (May 2025) 
• Data Warehousing Essentials and DMBOK – Coursera (Dec 2024) 
• Data Visualization Using Python – Infosys (Dec 2025) 
 
 
 
 
Skills 
 
              •            Programming & Tools: Python, R, MySQL, Streamlit 
              •            Data Handling: Data Cleaning, Wrangling, ETL Processes 
              •            Machine Learning: Linear Regression, Logistic Regression, Decision Trees, 
                             Random Forest, Support Vector Machines, Neural Networks, Unsupervised Learning 
              •            Advanced Techniques: Deep Learnin

In [48]:
# Convert the vector store into a retriever that can perform semantic search
retriever=vectorstore.as_retriever(search_kwargs={"k":2})
retriever_results=retriever.invoke("who is the candidate whose cv we are seeing?")
print(retriever_results)

[Document(id='ba2c9bf1-bfba-4493-b017-29b0d1bcf5a9', metadata={'creator': 'Microsoft® Word 2016', 'total_pages': 4, 'producer': 'www.ilovepdf.com', 'moddate': '2025-06-02T11:30:33+00:00', 'source': '/content/drive/MyDrive/project/LLM CHAT  FOR DOCS /docs/HRIDHIKJAL SB.pdf', 'page': 0, 'page_label': '1', 'author': 'HP', 'creationdate': '2025-06-02T11:30:33+00:00'}, page_content='• B.Sc. in Mathematics – SN College, Kannur, Kerala (May 2020) \n• Higher Secondary (Science) – PRMHSS, Kannur, Kerala (May 2018) \n \n \nCertifications \n• Data Science – NACTET (Aug 2024) \n• Business Analysis & Process Management – Coursera (May 2025) \n• Data Warehousing Essentials and DMBOK – Coursera (Dec 2024) \n• Data Visualization Using Python – Infosys (Dec 2025) \n \n \n \n \nSkills \n \n              •            Programming & Tools: Python, R, MySQL, Streamlit \n              •            Data Handling: Data Cleaning, Wrangling, ETL Processes \n              •            Machine Learning: Linear Reg

In [49]:
from re import template
from langchain_core.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Define a prompt template for the LLM to answer questions using only the retrieved context
template ="""

Answer the question based only on the following context :
{context}
Question: {question}
Answer: """

# Convert the raw string template into a structured ChatPromptTemplate object
prompt =ChatPromptTemplate.from_template(template)



# Define the RAG pipeline chain
rag_chain =(
    {"context":retriever,"question":RunnablePassthrough()}
    |prompt
    |llm
    |StrOutputParser()
)

In [50]:
#texting rag_chain

# question="which college does the person studies?"
# response=rag_chain.invoke(question)
# print(f"question: {question}")
# print(f"response: {response}")

In [55]:
from decimal import Context
from langchain_core.prompts import MessagesPlaceholder
from langchain.chains import create_history_aware_retriever
from langchain.chains.combine_documents import create_stuff_documents_chain

#System instruction for rephrasing a question using prior chat history
context_q_system_prompt="""
Given a chat history and latest user question
which might refernce context in chat history,
formulate a standalone quesion which can be understood without the chat history.
Do not answer the question,
just formulate it if needed and otherwise return as it is.
"""
# Define a prompt that includes the system instruction, chat history, and new user message
context_q_prompt = ChatPromptTemplate.from_messages([
    "system",context_q_system_prompt,
     MessagesPlaceholder(variable_name="chat_history"),
    ("human","{input}")
  ]
  )
#checking
context_chain=context_q_prompt | llm | StrOutputParser()
print(context_chain.invoke({"input":"what is your number","chat_history":[]}))




What is your phone number?


In [52]:

from langchain.chains import create_retrieval_chain

history_aware_retriever=create_history_aware_retriever(
    llm,retriever,context_q_prompt
)

qa_prompt=ChatPromptTemplate.from_messages([
     ("system","you are a helpfull AI assistant. use following context to answer the user's question."),
     ("system","Context:{context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human","{input}"),
])

question_answer_chain=create_stuff_documents_chain(llm,qa_prompt)
rag_chain=create_retrieval_chain(history_aware_retriever,question_answer_chain)

In [53]:
from langchain_core.messages import HumanMessage,AIMessage

chat_history =[]
question1 ="where is the college"
answer1=rag_chain.invoke({"input":question1,"chat_history":chat_history})['answer']

chat_history.extend([
    HumanMessage(content=question1),
    AIMessage(content=answer1)
])
chat_history.append(HumanMessage(content=question1))
chat_history.append(AIMessage(content=answer1))
print(question1)
print(answer1)


where is the college
SN College is located in Kannur, Kerala.
